PlantVillage Coursework — CNN Training + Metaheuristic Hyperparameter Optimisation (PyTorch + NiaPy)

What this script does
- Uses a CNN classifier (EfficientNet-B0 pretrained by default) for PlantVillage disease classification.
- Uses 8 nature-inspired / evolutionary algorithms (from NiaPy) as *hyperparameter optimisers*:
    1) Differential Evolution (DE)
    2) Particle Swarm Optimisation (PSO)
    3) Grey Wolf Optimiser (GWO)
    4) Artificial Bee Colony (ABC)
    5) Bat Algorithm (BA)
    6) Firefly Algorithm (FA)
    7) Cuckoo Search (CS)
    8) Coral Reefs Optimisation (CRO)

Important note (for your report)
- These 8 algorithms do NOT replace the CNN.
- They are used to search for good training hyperparameters (dropout, learning rates, weight decay)
  using validation Macro-F1 as the objective.
- After tuning, we train the CNN normally using backprop (AdamW / SGD).

Dataset structure expected (Kaggle "Plant Village Dataset (Updated)"):
    data_root/<Crop>/(train|val|test)/<ClassFolder>/*.jpg

Example:
    data/Apple/train/Healthy/xxx.jpg
    data/Apple/val/Healthy/yyy.jpg
    data/Apple/test/Healthy/zzz.jpg

Outputs (per run):
- runs_meta/<run_name>/run_config.json
- runs_meta/<run_name>/meta_tuning.json           (only if tuning enabled)
- runs_meta/<run_name>/history.json
- runs_meta/<run_name>/loss_curves.png
- runs_meta/<run_name>/macro_f1_curves.png
- runs_meta/<run_name>/pipeline_diagram.png
- runs_meta/<run_name>/best.pt
- runs_meta/<run_name>/last.pt
- runs_meta/<run_name>/test_metrics.json          (Accuracy, Precision, Sensitivity, F1, Computational Time)

Windows tip:
- keep NUM_WORKERS = 0 to avoid multiprocessing issues.

Dependencies:
- torch, torchvision, numpy, pillow, matplotlib, scikit-learn
- niapy  (pip install niapy)

Author: Avinesh & Avinash

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install niapy
!pip -q install kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.9/191.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 119.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
ERROR: pip's dependency resolver does not currentl

### Import lib & packages

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics import f1_score, precision_score, recall_score

import matplotlib.pyplot as plt

# ----------------------------
# NiaPy imports (8 algorithms)
# ----------------------------
try:
    from niapy.task import Task
    from niapy.problems import Problem

    from niapy.algorithms.basic.de import DifferentialEvolution
    from niapy.algorithms.basic.pso import ParticleSwarmOptimization
    from niapy.algorithms.basic.abc import ArtificialBeeColonyAlgorithm
    from niapy.algorithms.basic.gwo import GreyWolfOptimizer
    from niapy.algorithms.basic.ba import BatAlgorithm
    from niapy.algorithms.basic.fa import FireflyAlgorithm
    from niapy.algorithms.basic.cs import CuckooSearch
    from niapy.algorithms.basic.cro import CoralReefsOptimization

    NIAPY_AVAILABLE = True
except Exception as e:
    NIAPY_AVAILABLE = False
    NIAPY_IMPORT_ERROR = e

In [ ]:
print(f"is niapy available", NIAPY_AVAILABLE)

is niapy available True


In [ ]:
!pip -q install kaggle
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d tushar5harma/plant-village-dataset-updated -p /content/Data --unzip

Dataset URL: https://www.kaggle.com/datasets/tushar5harma/plant-village-dataset-updated
License(s): CC0-1.0
 98% 0.98G/1.00G [00:00<00:00, 1.68GB/s]
100% 1.00G/1.00G [00:00<00:00, 1.70GB/s]


In [ ]:
!find /content -maxdepth 2 -name "kaggle.json" -print

/content/kaggle.json


### User Setting

In [ ]:

DATA_ROOT = Path("/content/Data")
CROPS = ["Apple", "Bell Pepper", "Grape", "Peach", "Strawberry", "Tomato", "Potato", "Corn (Maize)"]  # folder names under DATA_ROOT

SEED = 42
# Use percentage sampling instead of fixed count
MAX_IMAGES_PER_CLASS = 0              # keep for fallback
MAX_IMAGES_PER_CLASS_PCT = 0.0

TUNE_MAX_IMAGES_PER_CLASS = 0         # keep for fallback
TUNE_MAX_IMAGES_PER_CLASS_PCT = 0.02

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 4

# Training controls (final training after tuning)
EPOCHS = 25
FREEZE_EPOCHS = 5
PATIENCE = 7
MONITOR = "macro_f1"  # "macro_f1" or "val_loss"
OPTIMIZER = "adamw"   # "adamw" or "sgd"
MOMENTUM = 0.9        # for SGD only
LOSS_FN = "cross_entropy"

# Defaults
DROPOUT = 0.3
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4

# Metaheuristic tuning
TUNE_ENABLED = True

TUNE_EPOCHS = 2
TUNE_FREEZE_EPOCHS = 1
TUNE_PATIENCE = 1

TUNE_POP_SIZE = 4
TUNE_ITERS = 2

# vector x = [dropout, log10(lr_head), log10(lr_finetune), log10(weight_decay)]
HP_LOWER = np.array([0.00, math.log10(1e-4), math.log10(1e-5), math.log10(1e-6)], dtype=float)
HP_UPPER = np.array([0.60, math.log10(3e-3), math.log10(5e-4), math.log10(1e-3)], dtype=float)

# Output folder
OUTPUT_ROOT = Path("./runs_meta")

EXPORT_TORCHSCRIPT = True
SAVE_DIAGRAMS = True

# 8 algorithm runs (all tune EfficientNet-B0 hyperparameters; same CNN backbone)
RUNS = [
    {"run_name": "effnet_de",      "model": "efficientnet_b0", "search_algo": "de",      "grayscale": True, "augment": True},
    {"run_name": "effnet_pso",     "model": "efficientnet_b0", "search_algo": "pso",     "grayscale": True, "augment": True},
    {"run_name": "effnet_gwo",     "model": "efficientnet_b0", "search_algo": "gwo",     "grayscale": True, "augment": True},
    {"run_name": "effnet_bat",     "model": "efficientnet_b0", "search_algo": "bat",     "grayscale": True, "augment": True},
    {"run_name": "effnet_firefly", "model": "efficientnet_b0", "search_algo": "fa",      "grayscale": True, "augment": True},
    {"run_name": "effnet_cuckoo",  "model": "efficientnet_b0", "search_algo": "cs",      "grayscale": True, "augment": True},
    {"run_name": "effnet_cro",     "model": "efficientnet_b0", "search_algo": "cro",     "grayscale": True, "augment": True},
    {"run_name": "effnet_abc",      "model": "efficientnet_b0", "search_algo": "abc",      "grayscale": True, "augment": True}
]

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


### HELPERS (REPRODUCIBILITY + SAMPLING)

In [ ]:


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # no-op on CPU

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def stable_int_seed(base_seed: int, crop: str, split: str, class_name: str) -> int:
    key = f"{base_seed}__{crop}__{split}__{class_name}".encode("utf-8")
    h = hashlib.md5(key).hexdigest()[:8]
    return int(h, 16)


def find_split_dir(crop_dir: Path, split: str) -> Path:
    s = split.lower()
    if s == "train":
        candidates = ["train", "Train", "TRAIN"]
    elif s == "val":
        candidates = ["val", "Val", "VAL", "validation", "Validation", "valid", "Valid"]
    elif s == "test":
        candidates = ["test", "Test", "TEST"]
    else:
        candidates = [split]

    for name in candidates:
        p = crop_dir / name
        if p.exists() and p.is_dir():
            return p

    raise FileNotFoundError(f"Split '{split}' not found under {crop_dir}. Expected one of: {candidates}")


def discover_class_names(data_root: Path, crops: Sequence[str]) -> List[str]:
    labels = set()
    for crop in crops:
        crop_dir = data_root / crop
        if not crop_dir.exists():
            raise FileNotFoundError(f"Crop folder not found: {crop_dir}")

        for split in ["train", "val", "test"]:
            try:
                split_dir = find_split_dir(crop_dir, split)
            except FileNotFoundError:
                continue

            for cls_dir in split_dir.iterdir():
                if cls_dir.is_dir():
                    labels.add(f"{crop}__{cls_dir.name}")

    if not labels:
        raise RuntimeError("No class folders found. Check your DATA_ROOT/CROPS and split structure.")

    return sorted(labels)


def collect_samples(
    data_root: Path,
    crops: Sequence[str],
    split: str,
    class_to_idx: Dict[str, int],
    max_images_per_class: int,
    seed: int,
    max_images_per_class_pct: float = 0.0,
) -> List[Tuple[str, int]]:
    """
    Collect (image_path, y) pairs for split.

    Sampling rule per class folder:
    - If max_images_per_class_pct > 0: take ceil(pct * N) images (at least 1)
    - Else if max_images_per_class > 0: take that many images
    - Else: take all images
    """
    samples: List[Tuple[str, int]] = []

    pct = float(max_images_per_class_pct or 0.0)
    if pct < 0 or pct > 1:
        raise ValueError("max_images_per_class_pct must be in [0, 1].")

    for crop in sorted(crops):
        crop_dir = data_root / crop
        split_dir = find_split_dir(crop_dir, split)

        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir():
                continue

            files = [p for p in sorted(class_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMG_EXTS]
            if not files:
                continue

            rng = random.Random(stable_int_seed(seed, crop, split, class_dir.name))
            rng.shuffle(files)

            # --- APPLY PERCENT OR COUNT ---
            if pct > 0.0:
                k = int(math.ceil(pct * len(files)))
                k = max(1, k)  # at least 1 per class
                files = files[:k]
            elif max_images_per_class and max_images_per_class > 0:
                files = files[:max_images_per_class]

            label_name = f"{crop}__{class_dir.name}"
            y = class_to_idx[label_name]
            for f in files:
                samples.append((str(f), y))

    if not samples:
        raise RuntimeError(f"No images found for split='{split}'. Check folder structure and extensions.")

    return samples

def save_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


### DATASET + TRANSFORMS

In [ ]:
class IdentityTransform:
    def __call__(self, x):
        return x


class LeafDataset(Dataset):
    def __init__(self, samples: List[Tuple[str, int]], transform):
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, y


def build_transforms(img_size: int, grayscale: bool, augment: bool):
    gray = transforms.Grayscale(num_output_channels=3) if grayscale else IdentityTransform()

    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )

    if augment:
        jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15) if grayscale else \
                 transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)

        train_tfm = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            gray,
            jitter,
            transforms.ToTensor(),
            normalize,
        ])
    else:
        train_tfm = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            gray,
            transforms.ToTensor(),
            normalize,
        ])

    eval_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        gray,
        transforms.ToTensor(),
        normalize,
    ])

    return train_tfm, eval_tfm


### MODELS

In [ ]:
def _get_weights_enum(model_name: str):
    try:
        if model_name == "efficientnet_b0":
            return models.EfficientNet_B0_Weights.DEFAULT
    except Exception:
        return None
    return None


def build_model(model_name: str, num_classes: int, dropout: float, pretrained: bool):
    if model_name == "efficientnet_b0":
        w = _get_weights_enum("efficientnet_b0")
        try:
            model = models.efficientnet_b0(weights=w if pretrained else None)
        except TypeError:
            model = models.efficientnet_b0(pretrained=pretrained)

        if isinstance(model.classifier, nn.Sequential):
            in_features = None
            for m in reversed(model.classifier):
                if isinstance(m, nn.Linear):
                    in_features = m.in_features
                    break
            if in_features is None:
                raise RuntimeError("Could not find Linear layer in EfficientNet classifier.")
            model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        else:
            raise RuntimeError("Unexpected EfficientNet classifier type; please update build_model.")

        return model, True

    raise ValueError(f"Unknown model: {model_name}")


def freeze_backbone(model: nn.Module, model_name: str) -> None:
    for p in model.parameters():
        p.requires_grad = False

    if model_name in {"efficientnet_b0"}:
        for p in model.classifier.parameters():
            p.requires_grad = True


def unfreeze_all(model: nn.Module) -> None:
    for p in model.parameters():
        p.requires_grad = True



### TRAINING (METRICS + EARLY STOPPING)

In [ ]:
@dataclass
class RunConfig:
    run_name: str
    model: str
    search_algo: str
    grayscale: bool
    augment: bool
    dropout: float
    seed: int
    max_images_per_class: int
    img_size: int
    batch_size: int
    epochs: int
    freeze_epochs: int
    patience: int
    monitor: str
    optimizer: str
    lr_head: float
    lr_finetune: float
    weight_decay: float
    momentum: float
    loss_fn: str
    crops: List[str]


class EarlyStopping:
    def __init__(self, patience: int, mode: str = "max"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.num_bad = 0

    def improved(self, value: float) -> bool:
        if self.best is None:
            return True
        return value > self.best if self.mode == "max" else value < self.best

    def step(self, value: float) -> bool:
        if self.improved(value):
            self.best = value
            self.num_bad = 0
            return False
        self.num_bad += 1
        return self.num_bad >= self.patience


def make_optimizer(model: nn.Module, name: str, lr: float, weight_decay: float, momentum: float):
    params = [p for p in model.parameters() if p.requires_grad]
    if name.lower() == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name.lower() == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


@torch.no_grad()
def run_eval(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    losses = []
    y_true, y_pred = [], []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)
        losses.append(loss.item() * yb.size(0))

        preds = torch.argmax(logits, dim=1)
        y_true.extend(yb.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    n = max(len(loader.dataset), 1)
    avg_loss = float(np.sum(losses) / n)

    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)

    acc = float(np.mean(y_true_np == y_pred_np))
    precision = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    sensitivity = float(recall_score(y_true, y_pred, average="macro", zero_division=0))  # recall
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "sensitivity": sensitivity,
        "macro_f1": macro_f1,
    }


def run_train_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer, device: torch.device) -> Dict[str, float]:
    model.train()
    losses = []
    y_true, y_pred = [], []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item() * yb.size(0))
        preds = torch.argmax(logits, dim=1)

        y_true.extend(yb.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    n = max(len(loader.dataset), 1)
    avg_loss = float(np.sum(losses) / n)

    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)

    acc = float(np.mean(y_true_np == y_pred_np))
    precision = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    sensitivity = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "sensitivity": sensitivity,
        "macro_f1": macro_f1,
    }


def plot_training_curves(history: List[Dict[str, float]], out_dir: Path) -> None:
    if not history:
        return

    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    train_f1 = [h["train_macro_f1"] for h in history]
    val_f1 = [h["val_macro_f1"] for h in history]

    out_dir.mkdir(parents=True, exist_ok=True)

    plt.figure()
    plt.plot(epochs, train_loss, label="train_loss")
    plt.plot(epochs, val_loss, label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "loss_curves.png", dpi=200)
    plt.close()

    plt.figure()
    plt.plot(epochs, train_f1, label="train_macro_f1")
    plt.plot(epochs, val_f1, label="val_macro_f1")
    plt.xlabel("epoch")
    plt.ylabel("macro_f1")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "macro_f1_curves.png", dpi=200)
    plt.close()


def save_pipeline_diagram(out_path: Path) -> None:
    plt.figure(figsize=(10, 4))
    ax = plt.gca()
    ax.axis("off")

    boxes = [
        ("Raw images\n(DATA_ROOT/<Crop>/<split>/<class>)", (0.05, 0.55)),
        ("Preprocess\nresize + normalize\n(optional grayscale)", (0.28, 0.55)),
        ("Augmentation\n(train only)", (0.50, 0.55)),
        ("CNN Model\n(EfficientNet-B0)\n+ dropout", (0.68, 0.55)),
        ("Train\n+ early stopping\n+ save best.pt", (0.83, 0.55)),
        ("Evaluate\n(test metrics)", (0.83, 0.20)),
    ]

    for text, (x, y) in boxes:
        ax.text(x, y, text, va="center", ha="center",
                bbox=dict(boxstyle="round", fc="white"))

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->"))

    arrow(0.12, 0.55, 0.22, 0.55)
    arrow(0.35, 0.55, 0.44, 0.55)
    arrow(0.57, 0.55, 0.62, 0.55)
    arrow(0.74, 0.55, 0.80, 0.55)
    arrow(0.86, 0.48, 0.86, 0.28)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

### METAHEURISTIC TUNING (NiaPy)

In [ ]:
def decode_hparams(x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x, dtype=float)

    dropout = float(np.clip(x[0], HP_LOWER[0], HP_UPPER[0]))
    lr_head = float(10 ** x[1])
    lr_finetune = float(10 ** x[2])
    weight_decay = float(10 ** x[3])

    lr_finetune = min(lr_finetune, lr_head)  # keep stage-2 LR <= stage-1 LR
    return {
        "dropout": dropout,
        "lr_head": lr_head,
        "lr_finetune": lr_finetune,
        "weight_decay": weight_decay,
    }


class HyperparamProblem(Problem):
    def __init__(self, objective_fn):
        super().__init__(dimension=HP_LOWER.size, lower=HP_LOWER, upper=HP_UPPER)
        self.objective_fn = objective_fn

    def _evaluate(self, x):
        return float(self.objective_fn(np.asarray(x, dtype=float)))


def build_metaheuristic(name: str):
    if not NIAPY_AVAILABLE:
        raise ImportError(f"NiaPy is not available. Install with: pip install niapy\nOriginal error: {NIAPY_IMPORT_ERROR}")

    name = name.lower().strip()

    if name == "de":
        return DifferentialEvolution(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "pso":
        return ParticleSwarmOptimization(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "abc":
        return ArtificialBeeColonyAlgorithm(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "gwo":
        return GreyWolfOptimizer(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "bat":
        return BatAlgorithm(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "fa":
        return FireflyAlgorithm(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "cs":
        return CuckooSearch(population_size=TUNE_POP_SIZE, seed=SEED)
    if name == "cro":
        return CoralReefsOptimization(population_size=TUNE_POP_SIZE, seed=SEED)

    raise ValueError(f"Unknown metaheuristic: {name}")


def tune_hyperparams(
    *,
    search_algo: str,
    model_name: str,
    grayscale: bool,
    augment: bool,
    class_names: List[str],
    class_to_idx: Dict[str, int],
    device: torch.device,
) -> Dict[str, Any]:
    """
    Tunes: dropout, lr_head, lr_finetune, weight_decay.
    Objective: MINIMIZE (1 - best_val_macro_f1).
    """

    # GPU ops are async; sync makes timings fairer
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    # Make a small subset for tuning
    train_samples = collect_samples(
    DATA_ROOT, CROPS, "train", class_to_idx,
    TUNE_MAX_IMAGES_PER_CLASS, SEED,
    max_images_per_class_pct=TUNE_MAX_IMAGES_PER_CLASS_PCT
    )
    val_samples = collect_samples(
        DATA_ROOT, CROPS, "val", class_to_idx,
        TUNE_MAX_IMAGES_PER_CLASS, SEED,
        max_images_per_class_pct=TUNE_MAX_IMAGES_PER_CLASS_PCT
    )

    train_tfm, eval_tfm = build_transforms(IMG_SIZE, grayscale=grayscale, augment=augment)
    train_ds = LeafDataset(train_samples, train_tfm)
    val_ds = LeafDataset(val_samples, eval_tfm)

    g = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, generator=g)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    cache: Dict[Tuple[float, float, float, float], float] = {}

    def objective(x: np.ndarray) -> float:
        # caching helps because metaheuristics sometimes revisit similar points
        key = tuple(np.round(x.astype(float), 4).tolist())
        if key in cache:
            return cache[key]

        hp = decode_hparams(x)

        set_seed(SEED)

        model, is_transfer = build_model(model_name, num_classes=len(class_names), dropout=hp["dropout"], pretrained=True)
        model.to(device)

        criterion = nn.CrossEntropyLoss().to(device)

        stopper = EarlyStopping(patience=TUNE_PATIENCE, mode="max")
        best_val_f1 = 0.0

        if is_transfer and TUNE_FREEZE_EPOCHS > 0:
            freeze_backbone(model, model_name)
            optimizer = make_optimizer(model, OPTIMIZER, hp["lr_head"], hp["weight_decay"], MOMENTUM)
        else:
            unfreeze_all(model)
            optimizer = make_optimizer(model, OPTIMIZER, hp["lr_finetune"], hp["weight_decay"], MOMENTUM)

        for epoch in range(1, TUNE_EPOCHS + 1):
            if is_transfer and epoch == TUNE_FREEZE_EPOCHS + 1:
                unfreeze_all(model)
                optimizer = make_optimizer(model, OPTIMIZER, hp["lr_finetune"], hp["weight_decay"], MOMENTUM)

            _ = run_train_epoch(model, train_loader, criterion, optimizer, device)
            va = run_eval(model, val_loader, criterion, device)

            best_val_f1 = max(best_val_f1, va["macro_f1"])
            if stopper.step(va["macro_f1"]):
                break

        # cleanup
        del model, optimizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

        fit = float(1.0 - best_val_f1)
        cache[key] = fit
        return fit

    algo = build_metaheuristic(search_algo)
    problem = HyperparamProblem(objective)
    task = Task(problem=problem, max_iters=TUNE_ITERS)  # minimization is default

    sync()
    t0 = time.perf_counter()
    best_x, best_fit = algo.run(task)
    sync()
    tune_time_sec = float(time.perf_counter() - t0)

    best_hp = decode_hparams(np.asarray(best_x, dtype=float))
    best_val_macro_f1 = float(1.0 - float(best_fit))

    return {
        "metaheuristic": search_algo,
        "best_hparams": best_hp,
        "best_val_macro_f1": best_val_macro_f1,
        "tune_time_sec": tune_time_sec,
        "tune_settings": {
            "tune_max_images_per_class": int(TUNE_MAX_IMAGES_PER_CLASS),
            "tune_epochs": int(TUNE_EPOCHS),
            "tune_freeze_epochs": int(TUNE_FREEZE_EPOCHS),
            "tune_patience": int(TUNE_PATIENCE),
            "population_size": int(TUNE_POP_SIZE),
            "iters": int(TUNE_ITERS),
            "search_dim": int(HP_LOWER.size),
        },
    }


### MAIN RUNNER

In [ ]:
def train_one_run(run: Dict) -> Dict[str, Any]:
    # Read run settings
    run_name = run["run_name"]
    model_name = run["model"]
    search_algo = str(run.get("search_algo", "none")).lower().strip()
    grayscale = bool(run.get("grayscale", False))
    augment = bool(run.get("augment", True))

    # Defaults (may be overwritten by tuning)
    dropout = float(run.get("dropout", DROPOUT))
    lr_head = float(run.get("lr_head", LR_HEAD))
    lr_finetune = float(run.get("lr_finetune", LR_FINETUNE))
    weight_decay = float(run.get("weight_decay", WEIGHT_DECAY))

    out_dir = OUTPUT_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    # Accurate timing on GPU
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    sync()
    t0_total = time.perf_counter()

    # ----------------------------
    # Data discovery
    # ----------------------------
    class_names = discover_class_names(DATA_ROOT, CROPS)
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    save_json(out_dir / "class_names.json", {"class_names": class_names})

    # ----------------------------
    # Metaheuristic tuning
    # ----------------------------
    tuning_summary = None
    if TUNE_ENABLED and search_algo != "none":
        print(f"\n[TUNING] run={run_name} algo={search_algo} ...")
        tuning_summary = tune_hyperparams(
            search_algo=search_algo,
            model_name=model_name,
            grayscale=grayscale,
            augment=augment,
            class_names=class_names,
            class_to_idx=class_to_idx,
            device=device,
        )
        save_json(out_dir / "meta_tuning.json", tuning_summary)

        # Override hyperparams with tuned values
        best_hp = tuning_summary["best_hparams"]
        dropout = float(best_hp["dropout"])
        lr_head = float(best_hp["lr_head"])
        lr_finetune = float(best_hp["lr_finetune"])
        weight_decay = float(best_hp["weight_decay"])

        print(
            f"[TUNING DONE] best_val_macro_f1={tuning_summary['best_val_macro_f1']:.4f} | "
            f"dropout={dropout:.3f} lr_head={lr_head:.2e} lr_finetune={lr_finetune:.2e} wd={weight_decay:.2e}"
        )

    # Save final chosen config
    cfg = RunConfig(
        run_name=run_name,
        model=model_name,
        search_algo=search_algo,
        grayscale=grayscale,
        augment=augment,
        dropout=dropout,
        seed=SEED,
        max_images_per_class=MAX_IMAGES_PER_CLASS,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        freeze_epochs=FREEZE_EPOCHS,
        patience=PATIENCE,
        monitor=MONITOR,
        optimizer=OPTIMIZER,
        lr_head=lr_head,
        lr_finetune=lr_finetune,
        weight_decay=weight_decay,
        momentum=MOMENTUM,
        loss_fn=LOSS_FN,
        crops=list(CROPS),
    )
    save_json(out_dir / "run_config.json", asdict(cfg))

    # ----------------------------
    # Build FULL data loaders (final train/val/test)
    # ----------------------------
    train_samples = collect_samples(DATA_ROOT, CROPS, "train", class_to_idx, MAX_IMAGES_PER_CLASS, SEED,
                                max_images_per_class_pct=MAX_IMAGES_PER_CLASS_PCT)
    val_samples   = collect_samples(DATA_ROOT, CROPS, "val",   class_to_idx, MAX_IMAGES_PER_CLASS, SEED,
                                    max_images_per_class_pct=MAX_IMAGES_PER_CLASS_PCT)
    test_samples  = collect_samples(DATA_ROOT, CROPS, "test",  class_to_idx, MAX_IMAGES_PER_CLASS, SEED,
                                    max_images_per_class_pct=MAX_IMAGES_PER_CLASS_PCT)


    train_tfm, eval_tfm = build_transforms(IMG_SIZE, grayscale=grayscale, augment=augment)
    train_ds = LeafDataset(train_samples, train_tfm)
    val_ds = LeafDataset(val_samples, eval_tfm)
    test_ds = LeafDataset(test_samples, eval_tfm)

    g = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, generator=g)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # ----------------------------
    # Model
    # ----------------------------
    model, is_transfer = build_model(model_name, num_classes=len(class_names), dropout=dropout, pretrained=True)
    model.to(device)
    print("model device:", next(model.parameters()).device)

    if LOSS_FN == "cross_entropy":
        criterion = nn.CrossEntropyLoss().to(device)
    else:
        raise ValueError(f"Unsupported LOSS_FN: {LOSS_FN}")

    es_mode = "max" if MONITOR == "macro_f1" else "min"
    stopper = EarlyStopping(patience=PATIENCE, mode=es_mode)

    best_path = out_dir / "best.pt"
    last_path = out_dir / "last.pt"

    # Stage 1: frozen backbone (transfer learning)
    if is_transfer and FREEZE_EPOCHS > 0:
        freeze_backbone(model, model_name)
        optimizer = make_optimizer(model, OPTIMIZER, lr_head, weight_decay, MOMENTUM)
    else:
        unfreeze_all(model)
        optimizer = make_optimizer(model, OPTIMIZER, lr_finetune, weight_decay, MOMENTUM)

    print(f"\n=== RUN: {run_name} ===")
    print(f"  CNN backbone : {model_name} (pretrained)")
    print(f"  Metaheuristic: {search_algo if search_algo != 'none' else 'none'}")
    print(f"  Grayscale    : {grayscale}")
    print(f"  Augment      : {augment}")
    print(f"  Device       : {device.type}")
    print(f"  Train/Val/Test sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")
    print(f"  Hyperparams: dropout={dropout:.3f} lr_head={lr_head:.2e} lr_finetune={lr_finetune:.2e} wd={weight_decay:.2e}")

    # ----------------------------
    # Train (timed)
    # ----------------------------
    history: List[Dict[str, float]] = []

    sync()
    t0_train = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        if is_transfer and epoch == FREEZE_EPOCHS + 1:
            unfreeze_all(model)
            optimizer = make_optimizer(model, OPTIMIZER, lr_finetune, weight_decay, MOMENTUM)

        tr = run_train_epoch(model, train_loader, criterion, optimizer, device)
        va = run_eval(model, val_loader, criterion, device)

        row = {
            "epoch": epoch,

            "train_loss": tr["loss"],
            "train_acc": tr["acc"],
            "train_precision": tr["precision"],
            "train_sensitivity": tr["sensitivity"],
            "train_macro_f1": tr["macro_f1"],

            "val_loss": va["loss"],
            "val_acc": va["acc"],
            "val_precision": va["precision"],
            "val_sensitivity": va["sensitivity"],
            "val_macro_f1": va["macro_f1"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"train acc {row['train_acc']:.4f} prec {row['train_precision']:.4f} sens {row['train_sensitivity']:.4f} f1 {row['train_macro_f1']:.4f} | "
            f"val acc {row['val_acc']:.4f} prec {row['val_precision']:.4f} sens {row['val_sensitivity']:.4f} f1 {row['val_macro_f1']:.4f}"
        )

        monitor_value = row["val_macro_f1"] if MONITOR == "macro_f1" else row["val_loss"]

        if stopper.improved(monitor_value):
            torch.save({
                "model": model.state_dict(),
                "epoch": epoch,
                "monitor": MONITOR,
                "monitor_value": float(monitor_value),
                "class_names": class_names,
                "run_config": asdict(cfg),
            }, best_path)

        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "class_names": class_names,
            "run_config": asdict(cfg),
        }, last_path)

        if stopper.step(monitor_value):
            print(f"Early stopping triggered (patience={PATIENCE}). Best {MONITOR}={stopper.best:.6f}")
            break

    sync()
    train_time_sec = float(time.perf_counter() - t0_train)

    # Save history + diagrams
    save_json(out_dir / "history.json", {"history": history})

    if SAVE_DIAGRAMS:
        plot_training_curves(history, out_dir)
        save_pipeline_diagram(out_dir / "pipeline_diagram.png")

    # ----------------------------
    # Load best model
    # ----------------------------
    try:
        ckpt = torch.load(best_path, map_location=device, weights_only=True)
    except TypeError:
        ckpt = torch.load(best_path, map_location=device)

    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    # Optional export (deployment)
    if EXPORT_TORCHSCRIPT:
        example = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        traced = torch.jit.trace(model, example)
        traced.save(str(out_dir / "best_torchscript.pt"))

    # ----------------------------
    # Test evaluation (timed)
    # ----------------------------
    sync()
    t0_test = time.perf_counter()
    test_metrics = run_eval(model, test_loader, criterion, device)
    sync()
    test_time_sec = float(time.perf_counter() - t0_test)

    sync()
    total_time_sec = float(time.perf_counter() - t0_total)

    tune_time_sec = float(tuning_summary["tune_time_sec"]) if tuning_summary else 0.0

    # Requested metrics (no confusion matrix)
    test_metrics_out = {
        "accuracy": float(test_metrics["acc"]),
        "precision": float(test_metrics["precision"]),
        "sensitivity": float(test_metrics["sensitivity"]),
        "f1_score": float(test_metrics["macro_f1"]),
        "computational_time_sec": float(total_time_sec),

        # extra breakdown (useful for your results table)
        "tune_time_sec": float(tune_time_sec),
        "train_time_sec": float(train_time_sec),
        "test_time_sec": float(test_time_sec),
    }

    save_json(out_dir / "test_metrics.json", {"test": test_metrics_out})

    print(f"\nSaved best model: {best_path}")
    print(f"Test metrics: {test_metrics_out}")
    print(f"Outputs saved to: {out_dir.resolve()}")

    return {
        "run_name": run_name,
        "model": model_name,
        "search_algo": search_algo,
        "test": test_metrics_out,
    }


In [ ]:
def main():
    set_seed(SEED)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT.resolve()}")

    for crop in CROPS:
        if not (DATA_ROOT / crop).exists():
            raise FileNotFoundError(f"Crop folder missing: {(DATA_ROOT / crop).resolve()}")

    if TUNE_ENABLED and not NIAPY_AVAILABLE:
        raise ImportError(
            f"TUNE_ENABLED=True but NiaPy could not be imported.\n"
            f"Install with: pip install niapy\n"
            f"Original error: {NIAPY_IMPORT_ERROR}"
        )

    summary = []
    for run in RUNS:
        result = train_one_run(run)
        summary.append(result)

    save_json(OUTPUT_ROOT / "summary_results.json", {"results": summary})
    print(f"\nSummary saved to: {(OUTPUT_ROOT / 'summary_results.json').resolve()}")


if __name__ == "__main__":
    main()


device: cuda

[TUNING] run=effnet_gwo algo=gwo ...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 98.5MB/s]


[TUNING DONE] best_val_macro_f1=0.8380 | dropout=0.107 lr_head=3.00e-03 lr_finetune=5.00e-04 wd=9.96e-04
model device: cuda:0

=== RUN: effnet_gwo ===
  CNN backbone : efficientnet_b0 (pretrained)
  Metaheuristic: gwo
  Grayscale    : True
  Augment      : True
  Device       : cuda
  Train/Val/Test sizes: 50181 / 11279 / 1265
  Hyperparams: dropout=0.107 lr_head=3.00e-03 lr_finetune=5.00e-04 wd=9.96e-04
Epoch 01 | train acc 0.8293 prec 0.8280 sens 0.8296 f1 0.8284 | val acc 0.9073 prec 0.9125 sens 0.9068 f1 0.9078
Epoch 02 | train acc 0.8760 prec 0.8755 sens 0.8764 f1 0.8758 | val acc 0.9092 prec 0.9179 sens 0.9091 f1 0.9103
Epoch 03 | train acc 0.8829 prec 0.8825 sens 0.8831 f1 0.8827 | val acc 0.9095 prec 0.9183 sens 0.9087 f1 0.9100
Epoch 04 | train acc 0.8859 prec 0.8854 sens 0.8861 f1 0.8857 | val acc 0.9099 prec 0.9178 sens 0.9096 f1 0.9113
Epoch 05 | train acc 0.8866 prec 0.8863 sens 0.8870 f1 0.8866 | val acc 0.9252 prec 0.9290 sens 0.9246 f1 0.9255
Epoch 06 | train acc 0.9400